# Spatial SVG benchmark — combined framework — Colab runner

Branch `spatial-svg-benchmark` of `wpuzhu/BIOINF-590-Individual-Project`. Method and results: `spatial_benchmark/FRAMEWORK.md`.

This notebook:
1. installs the pipeline (and, optionally, GPU domain methods: GraphST, STAGATE, SpaGCN);
2. downloads human DLPFC Visium slices (Maynard et al. 2021; 12 slices, 3 donors, manual layers) from Hugging Face, with SODB as fallback;
3. runs the combined framework and its baselines on real slices:
   * **R1** domains: single methods, majority vote (Kang 2025), STCC, EnSDD, PCE (ours) vs manual layers
   * **R2** slice-calibrated null (false calls), **R3** spike-in (recall), **R4** adjacent-section and cross-donor replication, **R5** perturbations, **R6** global SVG references
   * **Uncertainty**: per-spot confidence and replicate agreement vs manual layer boundaries; per-gene selection probability π; known layer markers
4. writes a self-contained `report.html` and zips all results.

Optional: re-run the held-out simulation validation of the paper figures (CPU, ~30–60 min).

**Before running:** *Runtime → Change runtime type → T4 GPU* (only needed for `gpu_quick` / `full`).

| Profile | Slices | Methods | Approx. time (T4) |
|---|---|---|---|
| `quick` | 151673 + 151674 | 5 CPU methods | ~45–60 min |
| `gpu_quick` | 151673 + 151674 | CPU + GraphST, STAGATE, SpaGCN | ~1.5–2 h |
| `full` | all 12 (6 adjacent pairs, 3 donors) | CPU + GPU, B = 20 | ~10–14 h (use Drive so it can resume) |

Results are checkpointed per slice. If Colab disconnects, re-run the notebook: finished slices are skipped.

In [ ]:
# ---- settings ----
PROFILE = "quick"          # "quick" | "gpu_quick" | "full"
USE_DRIVE = False          # True: save results to Google Drive (recommended for "full")
SAMPLES = None             # e.g. "151673,151674" to override the profile's slices
RUN_SIM_VALIDATION = False # True: also re-run the held-out simulation validation (CPU, ~30-60 min)
BRANCH = "spatial-svg-benchmark"
REPO = "https://github.com/wpuzhu/BIOINF-590-Individual-Project.git"

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU visible (fine for PROFILE='quick').")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = f"/content/drive/MyDrive/svg_benchmark/{PROFILE}"
else:
    RESULTS_DIR = f"/content/results/{PROFILE}"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("results ->", RESULTS_DIR)

In [ ]:
%cd /content
!rm -rf BIOINF-590-Individual-Project
!git clone -q --depth 1 -b $BRANCH $REPO
%cd /content/BIOINF-590-Individual-Project/spatial_benchmark
!git log --oneline -1

In [ ]:
# core dependencies (CPU pipeline + data download)
!pip install -q anndata scanpy huggingface_hub scikit-misc

In [ ]:
# optional GPU domain methods (skipped for PROFILE == "quick")
if PROFILE != "quick":
    import torch
    tv = torch.__version__.split("+")[0]
    cu = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
    print("torch", tv, cu)
    # GraphST (POT for optimal transport utilities); --no-deps avoids its old version pins
    !pip install -q POT
    !pip install -q --no-deps git+https://github.com/JinmiaoChenLab/GraphST.git
    # SpaGCN (we use its k-means initialisation, so louvain is not needed)
    !pip install -q --no-deps SpaGCN
    # STAGATE (PyG version): needs torch_geometric + torch_sparse wheels matching torch/CUDA
    !pip install -q torch_geometric
    !pip install -q torch_sparse -f https://data.pyg.org/whl/torch-{tv}+{cu}.html
    !pip install -q --no-deps git+https://github.com/QIFEIDKN/STAGATE_pyG.git
    import sys; sys.path.insert(0, ".")
    from stbench.real import gpu_methods as G
    ok, problems = G.available()
    print("GPU methods available:", list(ok))
    print("problems (these methods will be skipped):", problems)

In [ ]:
# sanity checks: unit tests + a 1-minute simulated run of the full pipeline
!python -m unittest discover -s tests 2>&1 | tail -3
!python scripts/run_real.py --config configs/sim_test.json --out /content/sim_check 2>&1 | grep -v -i warning | tail -4

In [ ]:
# data check: load one slice and confirm counts, coordinates and layer labels were found
import sys; sys.path.insert(0, ".")
from stbench.real import io
adata, src = io.load_dlpfc("151673", cache_dir="data")
print("source:", src, "| shape:", adata.shape)
print("obs columns:", list(adata.obs.columns)[:25])
print("label column detected:", io.find_label(adata))
ctx = io.to_context(adata)
print({k: ctx.extra[k] for k in ["count_source", "label_col", "n_spots", "n_genes", "label_names"]})
# If the label column is wrong, set "label_col" in configs/real_<profile>.json.
# If both download sources fail, upload DLPFC_151673.h5ad etc. to /content/BIOINF-590-Individual-Project/spatial_benchmark/data/

In [ ]:
# ---- the real-data run ----
extra = f"--samples {SAMPLES}" if SAMPLES else ""
!python scripts/run_real.py --config configs/real_{PROFILE}.json --out "$RESULTS_DIR" $extra 2>&1 | grep -v -i warning

In [ ]:
# optional: reproduce the simulation validation of the combined framework (results/combined + figures)
if RUN_SIM_VALIDATION:
    !python scripts/run_combined.py --out "$RESULTS_DIR/sim_validation" --workers 2 --B 20
    !python scripts/make_combined_figures.py "$RESULTS_DIR/sim_validation"

In [ ]:
from IPython.display import HTML, display
report = os.path.join(RESULTS_DIR, "report.html")
display(HTML(open(report).read()))

In [ ]:
# download everything (report + per-slice CSVs + gene lists)
import shutil
zip_path = shutil.make_archive(f"/content/svg_benchmark_{PROFILE}", "zip", RESULTS_DIR)
from google.colab import files
files.download(zip_path)

## What to look at / send back

* `report.html`: all results with interpretation notes, including section 7 (uncertainty).
* `<slice>/domain_svgs.csv`: per gene, selection probability `pi` (combined rule), `selected`, `modal_layer`, `domain_purity`, and the calls of every baseline (Kang pipeline, SplitStab strict v2, v1 stability, global lists).
* `<slice>/uncertainty_spots.csv`: per spot, PCE confidence, replicate agreement, correctness and distance to the manual layer boundary.
* `<slice>/marker_check.csv`: known layer markers, their π and assigned layer.
* `r4_replication.csv`, `r4_cross_donor.csv` (full profile): reproducibility between adjacent sections and between donors.

Download the zip and share it (or push it under `spatial_benchmark/results/`) to continue the analysis.